# Visual Production

Loads a canonical script and voiceover, obtains a generated, ChatGPT, or
fixture visual plan, resolves assets, renders short scene clips, and
assembles the final video with audio.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import os
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if not (PROJECT_ROOT / "src" / "pipeline").exists():
    raise RuntimeError("Run this notebook from the repository root or output directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline.config import ConfigRepository
from src.pipeline.orchestration import load_run_context, resolve_script_result, resolve_voiceover_package, validate_visual_plan_result
from src.pipeline.orchestration.config_templates import render_config_templates as render_template_values

OUTPUT_ROOT = PROJECT_ROOT / "output"
PIPELINE_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline" / "config" / "workbench.yaml"
CACHE_ROOT = PROJECT_ROOT / ".cache" / "ai_content_creation" / "models"
FIXTURE_DIR = PROJECT_ROOT / "tests" / "fixtures" / "media_roundup_reference"
IST = timezone(timedelta(hours=5, minutes=30))


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"Expected JSON object at {path}")
    return value


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_handoff_files(files: list[dict], output_dir: Path) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for file_info in files:
        filename = file_info["filename"]
        if "/" in filename or "\\" in filename:
            raise ValueError(f"Unsafe handoff filename: {filename}")
        path = output_dir / filename
        path.write_text(file_info["content"], encoding="utf-8")
        paths.append(str(path))
    return paths


def render_config_templates(value):
    context = {
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "OUTPUT_ROOT": str(OUTPUT_ROOT),
        "CACHE_ROOT": str(CACHE_ROOT),
        "RUN_ID": RUN_ID,
    }
    return render_template_values(value, context)


In [ ]:
# INPUT_MODE: "fixture" for deterministic smoke; "run" for a content run.
INPUT_MODE = "run"
RUN_ID_OR_CURRENT = "current"
SCRIPT_SOURCE = "auto"          # "auto", "app", or "chatgpt"
VOICEOVER_VARIANT = None           # Folder name/path; required only when multiple matching voiceovers exist.
VISUAL_PLAN_SOURCE = "generate" # "generate", "chatgpt", or "fixture"
VISUAL_RENDER_MODE = "preview"  # "preview" for iteration; "final" for delivery.
VISUAL_SCENE_IDS = []             # Optional focused rerender, for example ["scene_014"].
VISUAL_STORY_IDS = []             # Optional focused rerender by story id.

CONFIG_REPOSITORY = ConfigRepository(PIPELINE_CONFIG_PATH)
if INPUT_MODE == "fixture":
    RUN_ID = "fixture_media_roundup_reference"
    RUN_DIR = OUTPUT_ROOT / "tests" / "media_roundup_reference"
    VIDEO_PACKAGING_RESULT_PATH = FIXTURE_DIR / "video_packaging_result.json"
    SCRIPT_RESULT_PATH = FIXTURE_DIR / "script_generation_result.json"
    FIXTURE_VISUAL_PLAN_PATH = FIXTURE_DIR / "visual_scene_planning_result.json"
    RUN_PROFILE = CONFIG_REPOSITORY.default_profile
    script_generation_result = load_json(SCRIPT_RESULT_PATH)
else:
    context = load_run_context(output_root=OUTPUT_ROOT, run_id=RUN_ID_OR_CURRENT)
    RUN_ID = context["run_id"]
    RUN_DIR = Path(context["run_dir"])
    RUN_PROFILE = context["run_profile"]
    VIDEO_PACKAGING_RESULT_PATH = RUN_DIR / "video_packaging" / "video_packaging_result.json"
    SCRIPT_RESULT_PATH, script_generation_result = resolve_script_result(run_dir=RUN_DIR, run_id=RUN_ID, source=SCRIPT_SOURCE)
    FIXTURE_VISUAL_PLAN_PATH = FIXTURE_DIR / "visual_scene_planning_result.json"

PROFILE_CONFIG = CONFIG_REPOSITORY.profile(RUN_PROFILE).value


def visual_module_config(module_name: str) -> dict:
    return CONFIG_REPOSITORY.module(RUN_PROFILE, "visuals", module_name).value


VISUALS_DIR = RUN_DIR / "visuals"
visual_input_dir = RUN_DIR / "inputs" / "visuals"
video_packaging_result = load_json(VIDEO_PACKAGING_RESULT_PATH)
script_provenance = script_generation_result["data"]["script_package"]["provenance"]
voiceover_package_path, voiceover_package = resolve_voiceover_package(run_dir=RUN_DIR, run_id=RUN_ID, script_fingerprint=script_provenance["script_fingerprint"], variant=VOICEOVER_VARIANT)
voiceover_provenance = voiceover_package["provenance"]
voiceover_result_path = voiceover_package_path.parent / "voiceover_result.json"
voiceover_result = load_json(voiceover_result_path) if voiceover_result_path.exists() else {"status": "ok", "data": {"run_id": RUN_ID, "voiceover_package": voiceover_package}, "summary": {"audio_path": voiceover_package.get("audio", {}).get("full_audio_path", "")}, "handoff": voiceover_package.get("handoff", {})}
write_json(RUN_DIR / "orchestration" / "visual_selection.json", {"script_result_path": str(SCRIPT_RESULT_PATH), "script_fingerprint": script_provenance["script_fingerprint"], "voiceover_package_path": str(voiceover_package_path), "voiceover_fingerprint": voiceover_provenance["voiceover_fingerprint"], "visual_plan_source": VISUAL_PLAN_SOURCE})
print(f"Input mode:      {INPUT_MODE}")
print(f"Run ID:          {RUN_ID}")
print(f"Script:          {SCRIPT_RESULT_PATH}")
print(f"Voiceover:       {voiceover_package_path}")
print(f"Visual plan mode:{VISUAL_PLAN_SOURCE}")


## 1. Build Trusted Assets And Load Or Generate Visual Plan


In [ ]:
from src.pipeline.visuals.asset_catalog import run as build_visual_asset_catalog
from src.pipeline.visuals.planner import run as run_visual_scene_planner
from src.pipeline.visuals.planner.retime import bind_visual_plan_result, retime_visual_plan_result

asset_catalog_authoring = visual_module_config("asset_catalog")
if not isinstance(asset_catalog_authoring, dict):
    raise RuntimeError("This frozen run profile predates trusted visual asset catalogs. Start a fresh content_pipeline run with the current workbench config.")
asset_catalog_config = render_config_templates(deepcopy(asset_catalog_authoring))
asset_catalog_dir = VISUALS_DIR / "asset_catalog"
asset_catalog_config["output_dir"] = str(asset_catalog_dir)
if INPUT_MODE == "fixture":
    asset_catalog_config["page_metadata"]["enabled"] = False
    asset_catalog_config["contact_sheet"]["enabled"] = False
visual_asset_catalog_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": asset_catalog_config,
    "input": {
        "script_generation": script_generation_result,
        "video_packaging": video_packaging_result,
    },
}
visual_asset_catalog_result = build_visual_asset_catalog(visual_asset_catalog_input)
write_json(asset_catalog_dir / "visual_asset_catalog_input.json", visual_asset_catalog_input)
write_json(asset_catalog_dir / "visual_asset_catalog_result.json", visual_asset_catalog_result)
write_json(asset_catalog_dir / "summary.json", visual_asset_catalog_result.get("summary", {}))
if visual_asset_catalog_result.get("status") != "ok":
    raise RuntimeError(visual_asset_catalog_result.get("summary", {}))

planner_config = render_config_templates(visual_module_config("planner"))
planning_dir = VISUALS_DIR / "planning" / VISUAL_PLAN_SOURCE

if VISUAL_PLAN_SOURCE == "generate":
    visual_scene_planning_input = {
        "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
        "config": planner_config,
        "input": {
            "video_packaging": video_packaging_result,
            "script_generation": script_generation_result,
            "voiceover": voiceover_package,
            "asset_catalog": visual_asset_catalog_result,
        },
    }
    visual_scene_planning_result = run_visual_scene_planner(visual_scene_planning_input)
elif VISUAL_PLAN_SOURCE == "chatgpt":
    source_path = VISUALS_DIR / "planning" / "chatgpt" / "visual_scene_planning_result.json"
    visual_scene_planning_result = load_json(source_path)
    visual_scene_planning_input = {"source": str(source_path)}
elif VISUAL_PLAN_SOURCE == "fixture":
    visual_scene_planning_result = load_json(FIXTURE_VISUAL_PLAN_PATH)
    visual_scene_planning_input = {"source": str(FIXTURE_VISUAL_PLAN_PATH)}
else:
    raise ValueError(f"Unknown VISUAL_PLAN_SOURCE: {VISUAL_PLAN_SOURCE}")

planning_dir.mkdir(parents=True, exist_ok=True)
write_json(planning_dir / "visual_scene_planning_input.json", visual_scene_planning_input)
write_json(planning_dir / "visual_scene_planning_result.json", visual_scene_planning_result)
write_json(planning_dir / "summary.json", visual_scene_planning_result.get("summary", {}))
if visual_scene_planning_result.get("status") != "ok":
    failed_story_plans = [
        plan
        for plan in visual_scene_planning_result.get("data", {}).get("visual_scene_plan", {}).get("story_plans", [])
        if plan.get("planning_failed") or plan.get("fallback_used")
    ]
    failed_story_ids = {plan.get("story_id") for plan in failed_story_plans if plan.get("story_id")}
    failed_calls = [
        call
        for call in visual_scene_planning_result.get("llm_calls", [])
        if call.get("module_name") == "story_visual_planner"
        and call.get("metadata", {}).get("story_id") in failed_story_ids
    ]
    diagnostic_path = planning_dir / "failed_story_planner_diagnostics.json"
    write_json(diagnostic_path, {"summary": visual_scene_planning_result.get("summary", {}), "story_plans": failed_story_plans, "calls": failed_calls})
    failure_reason = visual_scene_planning_result.get("summary", {}).get("failure_reason", "visual planning failed")
    raise RuntimeError(f"{failure_reason}. Diagnostics: {diagnostic_path}")

voice_pkg = voiceover_package.get("data", {}).get("voiceover_package", voiceover_package)
audio = voice_pkg.get("audio", {})
audio_duration = float(audio.get("duration_seconds") or voice_pkg.get("duration_seconds") or 0)
plan_duration = float(visual_scene_planning_result["data"]["visual_scene_plan"]["scene_timeline"][-1]["end_seconds"])
if audio_duration > 0 and abs(audio_duration - plan_duration) > 0.05:
    if VISUAL_PLAN_SOURCE == "fixture":
        visual_scene_planning_result = retime_visual_plan_result(visual_scene_planning_result, audio_duration)
        print(f"Retimed reference plan: {plan_duration:.2f}s -> {audio_duration:.2f}s")
    else:
        raise RuntimeError(f"Visual plan duration {plan_duration:.2f}s does not match voiceover {audio_duration:.2f}s")

if VISUAL_PLAN_SOURCE == "fixture":
    visual_scene_planning_result = bind_visual_plan_result(visual_scene_planning_result, run_id=RUN_ID, script_provenance=script_provenance, voiceover_provenance=voiceover_provenance)
visual_scene_plan = validate_visual_plan_result(visual_scene_planning_result, run_id=RUN_ID, script_fingerprint=script_provenance["script_fingerprint"], voiceover_fingerprint=voiceover_provenance["voiceover_fingerprint"])
write_json(planning_dir / "visual_scene_plan.json", visual_scene_plan)
print(f"Scenes:   {len(visual_scene_plan['scene_timeline'])}")
print(f"Duration: {visual_scene_plan['scene_timeline'][-1]['end_seconds']}s")
print(f"Plan:     {planning_dir / 'visual_scene_planning_result.json'}")


In [ ]:
failed_story_plans = [
    plan
    for plan in visual_scene_planning_result.get("data", {}).get("visual_scene_plan", {}).get("story_plans", [])
    if plan.get("planning_failed") or plan.get("fallback_used")
]
failed_story_ids = {plan.get("story_id") for plan in failed_story_plans if plan.get("story_id")}

failed_calls = [
    call
    for call in visual_scene_planning_result.get("llm_calls", [])
    if call.get("module_name") == "story_visual_planner"
    and call.get("metadata", {}).get("story_id") in failed_story_ids
]

diagnostic_path = planning_dir / "failed_story_planner_diagnostics.json"

write_json(
    diagnostic_path,
    {
        "summary": visual_scene_planning_result.get("summary", {}),
        "story_plans": failed_story_plans,
        "calls": failed_calls,
    },
)

print(diagnostic_path if failed_story_plans else "No failed story plans in the current result.")

## 2. Discover Media And Resolve Assets


In [ ]:
visual_media_discovery_config = render_config_templates(visual_module_config("media_discovery"))
visual_asset_resolver_config = render_config_templates(visual_module_config("asset_resolver"))
if not visual_media_discovery_config:
    raise KeyError("Workbench profile is missing visuals.media_discovery config")
if not visual_asset_resolver_config:
    raise KeyError("Workbench profile is missing visuals.asset_resolver config")

visual_media_discovery_output_dir = VISUALS_DIR / "media_discovery"
visual_asset_resolution_output_dir = VISUALS_DIR / "asset_resolution"
visual_media_discovery_config["output_dir"] = str(visual_media_discovery_output_dir)
visual_asset_resolver_config["output_dir"] = str(visual_asset_resolution_output_dir)
if not visual_asset_resolver_config.get("logos", {}).get("local_logo_dir"):
    visual_asset_resolver_config.setdefault("logos", {})["local_logo_dir"] = str(PROJECT_ROOT / "src" / "pipeline" / "visuals" / "remotion_engine" / "assets" / "logos")

if "visual_scene_planning_result" not in globals():
    visual_scene_planning_result = load_json(VISUALS_DIR / "planning" / VISUAL_PLAN_SOURCE / "visual_scene_planning_result.json")
if visual_scene_planning_result.get("status") != "ok":
    raise RuntimeError(visual_scene_planning_result.get("summary", {}))

visual_media_discovery_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_media_discovery_config,
    "input": {"visual_scene_planning": visual_scene_planning_result},
}
visual_media_discovery_input_path = visual_input_dir / f"{RUN_ID}_visual_media_discovery_input.json"
write_json(visual_media_discovery_input_path, visual_media_discovery_input)

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
visual_media_discovery_process = subprocess.run(
    [sys.executable, "-m", "src.pipeline.visuals.media_discovery.cli", "--input-json", str(visual_media_discovery_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=False,
)
visual_media_discovery_result_path = visual_media_discovery_output_dir / "visual_media_discovery_result.json"
visual_media_discovery_summary_path = visual_media_discovery_output_dir / "summary.json"
if not visual_media_discovery_result_path.exists() or not visual_media_discovery_summary_path.exists():
    raise RuntimeError(f"Media discovery exited with code {visual_media_discovery_process.returncode} without writing its result package.")
visual_media_discovery_result = load_json(visual_media_discovery_result_path)
visual_media_discovery_summary = load_json(visual_media_discovery_summary_path)
media_llm_calls = visual_media_discovery_result.get("llm_calls", [])
(visual_media_discovery_output_dir / "llm_calls.jsonl").write_text("\n".join(json.dumps(item, ensure_ascii=False) for item in media_llm_calls) + ("\n" if media_llm_calls else ""), encoding="utf-8")
if visual_media_discovery_process.returncode != 0 or not visual_media_discovery_result.get("handoff", {}).get("ready_for_visual_asset_resolution", False):
    failure_reason = visual_media_discovery_summary.get("failure_reason", "media discovery did not produce a usable package")
    raise RuntimeError(f"Media discovery failed: {failure_reason}. Details: {visual_media_discovery_summary_path}")

visual_asset_resolution_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_asset_resolver_config,
    "input": {
        "visual_scene_planning": visual_scene_planning_result,
        "visual_media_discovery": visual_media_discovery_result,
    },
}

visual_asset_resolution_input_path = visual_input_dir / f"{RUN_ID}_visual_asset_resolution_input.json"
write_json(visual_asset_resolution_input_path, visual_asset_resolution_input)

subprocess.run(
    [sys.executable, "-m", "src.pipeline.visuals.asset_resolver.cli", "--input-json", str(visual_asset_resolution_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

visual_asset_resolution_result = load_json(visual_asset_resolution_output_dir / "visual_asset_resolution_result.json")
visual_asset_manifest = load_json(visual_asset_resolution_output_dir / "visual_asset_manifest.json")
visual_asset_resolution_summary = load_json(visual_asset_resolution_output_dir / "summary.json")

print(f"Media discovery status: {visual_media_discovery_result['status']}")
print(f"Selected images:        {visual_media_discovery_summary['selected_image_count']}")
print(f"Selected videos:        {visual_media_discovery_summary['selected_video_count']}")
print(f"Explicit fallbacks:     {visual_media_discovery_summary['fallback_scene_count']}")
print(f"Download errors:        {visual_media_discovery_summary.get('download_error_count', 0)}")
print(f"Visually verified:      {visual_media_discovery_summary.get('semantic_verified_candidate_count', 0)}")
print(f"Verifier accepted:      {visual_media_discovery_summary.get('semantic_accepted_candidate_count', 0)}")
print(f"Verifier rejected:      {visual_media_discovery_summary.get('semantic_rejected_candidate_count', 0)}")
print(f"Visual asset status:    {visual_asset_resolution_result['status']}")
print(f"Resolved assets:        {visual_asset_resolution_summary['resolved_asset_count']}")
print(f"Source screenshots:     {visual_asset_resolution_summary['source_screenshot_count']}")
print(f"Logo assets:            {visual_asset_resolution_summary['logo_asset_count']}")
print(f"Context images:         {visual_asset_resolution_summary.get('context_image_count', 0)}")
print(f"B-roll videos:          {visual_asset_resolution_summary.get('broll_video_count', 0)}")
print(f"Failed assets:          {visual_asset_resolution_summary['failed_asset_count']}")
print(f"Output dir:             {visual_asset_resolution_output_dir}")
if not visual_asset_resolution_result.get("handoff", {}).get("ready_for_visual_render", False):
    raise RuntimeError(visual_asset_resolution_summary)


In [ ]:
visual_asset_resolution_summary


## 3. Render Scene Clips


In [ ]:
visual_render_config = render_config_templates(visual_module_config("render"))
if not visual_render_config:
    raise KeyError("Workbench profile is missing visuals.render config")

render_profiles = visual_render_config.pop("profiles", {})
active_render_mode = VISUAL_RENDER_MODE or visual_render_config.pop("active_mode", "preview")
visual_render_config.pop("active_mode", None)
if active_render_mode not in render_profiles:
    raise KeyError(f"Unknown visual render mode {active_render_mode!r}; expected one of {sorted(render_profiles)}")
visual_render_config.update(render_profiles[active_render_mode])
visual_render_config["selected_scene_ids"] = list(VISUAL_SCENE_IDS)
visual_render_config["selected_story_ids"] = list(VISUAL_STORY_IDS)

visual_render_output_dir = VISUALS_DIR / ("render" if active_render_mode == "final" else "render_preview")
visual_render_config["output_dir"] = str(visual_render_output_dir)
if not visual_render_config.get("remotion_engine_dir"):
    visual_render_config["remotion_engine_dir"] = str(PROJECT_ROOT / "src" / "pipeline" / "visuals" / "remotion_engine")

if "visual_asset_resolution_result" not in globals():
    visual_asset_resolution_result = load_json(VISUALS_DIR / "asset_resolution" / "visual_asset_resolution_result.json")
if visual_asset_resolution_result.get("status") not in {"ok", "partial"}:
    raise RuntimeError(visual_asset_resolution_result.get("summary", {}))

visual_render_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_render_config,
    "input": {"visual_asset_resolution": visual_asset_resolution_result},
}

visual_render_input_dir = visual_input_dir
visual_render_input_path = visual_render_input_dir / f"{RUN_ID}_visual_render_{active_render_mode}_input.json"
write_json(visual_render_input_path, visual_render_input)

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [sys.executable, "-m", "src.pipeline.visuals.render.cli", "--input-json", str(visual_render_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

visual_render_result = load_json(visual_render_output_dir / "visual_render_result.json")
visual_render_package = load_json(visual_render_output_dir / "visual_render_package.json")
visual_render_summary = load_json(visual_render_output_dir / "render_summary.json")

print(f"Visual render status: {visual_render_result['status']}")
print(f"Render mode:          {active_render_mode}")
print(f"Rendered scenes:      {visual_render_summary['rendered_scene_count']}")
print(f"Cached scenes:        {visual_render_summary.get('cached_scene_count', 0)}")
print(f"Failed scenes:        {visual_render_summary['failed_scene_count']}")
print(f"Output dir:           {visual_render_output_dir}")
if visual_render_summary.get("failed_scene_count", 0):
    raise RuntimeError(visual_render_summary)
if not visual_render_summary.get("render_scope_complete", True):
    print("Focused render complete. Clear VISUAL_SCENE_IDS/VISUAL_STORY_IDS and rerun this cell before assembly.")


In [ ]:
visual_render_summary

## 4. Assemble Preview Or Final Video With Voiceover


In [ ]:
visual_assembly_config = render_config_templates(visual_module_config("assembly"))
if not visual_assembly_config:
    raise KeyError("Workbench profile is missing visuals.assembly config")

if "visual_render_result" not in globals():
    selected_render_mode = VISUAL_RENDER_MODE or "preview"
    selected_render_dir = VISUALS_DIR / ("render" if selected_render_mode == "final" else "render_preview")
    visual_render_result = load_json(selected_render_dir / "visual_render_result.json")

render_package = visual_render_result.get("data", {}).get("visual_render_package", {})
render_mode = render_package.get("render_settings", {}).get("render_mode", "final")
render_summary = visual_render_result.get("summary", {})
if not render_summary.get("render_scope_complete", True):
    raise RuntimeError("Focused scene/story renders cannot be assembled. Clear VISUAL_SCENE_IDS and VISUAL_STORY_IDS, then rerun section 3.")
visual_assembly_output_dir = VISUALS_DIR / ("assembly" if render_mode == "final" else "assembly_preview")
visual_assembly_config["output_dir"] = str(visual_assembly_output_dir)

if "voiceover_package" not in globals():
    script_provenance = script_generation_result["data"]["script_package"]["provenance"]
    voiceover_package_path, voiceover_package = resolve_voiceover_package(run_dir=RUN_DIR, run_id=RUN_ID, script_fingerprint=script_provenance["script_fingerprint"], variant=VOICEOVER_VARIANT)
    print(f"Loaded matching voiceover package: {voiceover_package_path}")

visual_assembly_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_assembly_config,
    "input": {
        "visual_render": visual_render_result,
        "voiceover": voiceover_package,
    },
}

visual_assembly_input_dir = visual_input_dir
visual_assembly_input_path = visual_assembly_input_dir / f"{RUN_ID}_visual_assembly_{render_mode}_input.json"
write_json(visual_assembly_input_path, visual_assembly_input)

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [sys.executable, "-m", "src.pipeline.visuals.assembly.cli", "--input-json", str(visual_assembly_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

video_assembly_result = load_json(visual_assembly_output_dir / "video_assembly_result.json")
final_video_package = load_json(visual_assembly_output_dir / "final_video_package.json")
video_assembly_summary = load_json(visual_assembly_output_dir / "assembly_summary.json")

print(f"Video assembly status: {video_assembly_result['status']}")
print(f"Assembly mode:         {render_mode}")
print(f"Scene clips:           {video_assembly_summary['assembled_scene_count']}")
print(f"Audio duration:        {video_assembly_summary['audio_duration_seconds']}s")
print(f"Final duration:        {video_assembly_summary['final_duration_seconds']}s")
print(f"Final video:           {video_assembly_summary['final_video_path']}")
if not video_assembly_result.get("handoff", {}).get("ready_for_manual_review", False):
    raise RuntimeError(video_assembly_summary)


In [ ]:
video_assembly_summary


## 5. Build Visual Contact Sheet

Samples the assembled video for a fast human check of relevance, identity, readability, and repetition.


In [ ]:
visual_review_config = render_config_templates(visual_module_config("review"))
if not visual_review_config:
    raise KeyError("Workbench profile is missing visuals.review config")
visual_review_output_dir = VISUALS_DIR / ("review" if render_mode == "final" else "review_preview")
visual_review_config["output_dir"] = str(visual_review_output_dir)
visual_review_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": visual_review_config,
    "input": {"video_assembly": video_assembly_result},
}
visual_review_input_path = visual_input_dir / f"{RUN_ID}_visual_review_{render_mode}_input.json"
write_json(visual_review_input_path, visual_review_input)
subprocess.run(
    [sys.executable, "-m", "src.pipeline.visuals.review.cli", "--input-json", str(visual_review_input_path)],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)
visual_review_result = load_json(visual_review_output_dir / "visual_review_result.json")
visual_review_summary = load_json(visual_review_output_dir / "summary.json")
if not visual_review_result.get("handoff", {}).get("ready_for_human_visual_review", False):
    raise RuntimeError(visual_review_summary)
print(f"Contact sheet: {visual_review_summary['contact_sheet_path']}")


## 6. Build Manual Publish Review Handoff

Requires the finalized audio-timed metadata package and the assembled video. It never publishes automatically.


In [ ]:
from src.pipeline.publishing.publish_handoff import run as build_publish_handoff

if INPUT_MODE == "fixture":
    print("Publish handoff skipped for fixture mode.")
elif render_mode != "final":
    print("Publish handoff skipped for preview mode. Switch VISUAL_RENDER_MODE to 'final' and rerun sections 3-6 when the preview is approved.")
else:
    metadata_finalizer_path = RUN_DIR / "metadata_finalizer" / "metadata_finalizer_result.json"
    if not metadata_finalizer_path.exists():
        raise FileNotFoundError(f"Finalized metadata not found: {metadata_finalizer_path}. Run voiceover_generation through metadata finalization first.")
    metadata_finalizer_result = load_json(metadata_finalizer_path)
    publish_handoff_input = {
        "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
        "config": {"require_ready_inputs": True},
        "input": {
            "run_context": context,
            "video_packaging": video_packaging_result,
            "script_generation": script_generation_result,
            "metadata_finalizer": metadata_finalizer_result,
            "voiceover": voiceover_result,
            "video_assembly": video_assembly_result,
            "visual_review": visual_review_result,
        },
    }
    publish_handoff_result = build_publish_handoff(publish_handoff_input)
    if publish_handoff_result["status"] != "ok":
        raise RuntimeError(publish_handoff_result["summary"])
    publish_dir = RUN_DIR / "handoffs" / "publish_review"
    write_json(publish_dir / "publish_handoff_input.json", publish_handoff_input)
    write_json(publish_dir / "publish_handoff_result.json", publish_handoff_result)
    written = write_handoff_files(publish_handoff_result["data"]["publish_handoff_package"]["files"], publish_dir)
    print(f"Publish review: {written[0]}")
